# Nomogram Construction

Libraries/Packages

In [ ]:
import sys
import os

sys.path.append(os.path.abspath("../"))
from src.data_utils import get_data, get_models, get_feature_lists
from src.nomogram import nomogram
import pandas as pd
import joblib
from src.config import SEED, BASE_PATH
from shutil import rmtree

Data + Model

In [ ]:
## Data
data_dict = get_data(is_nomo=True)
X_train = data_dict["X_train"]
y_train = data_dict["y_train"]
## Model
nomo_clf = get_models(["lr"])["lr"]
## Threshold (0.371)
class_report_df = pd.read_excel(
    BASE_PATH / "results/tables/metrics/class_report.xlsx", index_col=0
)
threshold = float(class_report_df.loc["lr"]["Threshold"])  # type: ignore

NOMO_PATH = BASE_PATH / "results" / "Nomogram"

Initialize feature lists

In [ ]:
##Imported func from src
feature_lists = get_feature_lists(X_train)
binary_cols = feature_lists["Binary"]
numerical_cols = feature_lists["Numerical"]
nominal_cols = feature_lists["Nominal"]
ordinal_cols = feature_lists["Ordinal"]

Run Nomogram

In [ ]:
#### Set up path
if NOMO_PATH.exists():
    rmtree(NOMO_PATH)
NOMO_PATH.mkdir(exist_ok=True, parents=True)
##Get coef and intercept from log model
coefs = nomo_clf.coef_[0]
intercept = nomo_clf.intercept_[0]
##Create features and coef columns

feature_names = X_train.columns.to_list()
features = ["intercept"] + ["threshold"] + feature_names
coefs = [intercept] + [0.502] + list(coefs)

##Get continuous, nominal, and ordinal columns
df_size = X_train.shape[1]
cat_cols = [0] * df_size  ## 0 = numerical
for i in range(0, df_size):
    col = X_train.columns[i]
    counts = X_train[col].value_counts()
    if col in nominal_cols or col in binary_cols:  # 1 = #Nominal
        cat_cols[i] = 1
    elif col in ordinal_cols:  ##2 = Ordinal
        cat_cols[i] = 2
##Get mins, maxs, coefs, types, positions columns
mins = [""] + [""] + [X_train[feature].min() for feature in feature_names]
maxs = [""] + [""] + [X_train[feature].max() for feature in feature_names]
coefs = [1e-10 if i == 0 else i for i in coefs]  # Make 0 entries a very small number
types = (
    [""]
    + [""]
    + ["numerical" if i == 0 else "nominal" if i == 1 else "ordinal" for i in cat_cols]
)
positions = [""] * (df_size + 2)
##Create table
nomo_table = pd.DataFrame(
    {
        "feature": features,
        "coef": coefs,
        "min": mins,
        "max": maxs,
        "type": types,
        "position": positions,
    }
)
##Export and import table
path_to_nomo_table = NOMO_PATH / "nomo_table.xlsx"
nomo_table.to_excel(path_to_nomo_table, index=False)
##Create nomogram
results = nomogram(
    path=path_to_nomo_table,
    result_title="ORN Risk",
    fig_width=30,
    single_height=0.7,
    dpi=500,
    ax_para={"linewidth": 2, "c": "black", "linestyle": "--"},
    xtick_para={"fontsize": 13},
)
results.savefig(NOMO_PATH / "nomogram.pdf", bbox_inches="tight", dpi=500)